In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from pyproj import Geod

from snail.damage_library import get_metadata, load_curve
from snail.damages import calculate_direct_damages
from snail.data import download_file
from snail.overlay import overlay_rasters

# Support running Jupyter from either the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent

iso3 = "KEN"
# See https://github.com/nismod/irv-datapkg/blob/main/records.csv for records
record_lookup = {
    "LCA": 23270777,
    "KEN": 23271131,
}
data_root = repo_root / "data"
url = f"https://zenodo.org/records/{record_lookup[iso3]}/files/{iso3}.zip?download=1"
download_file(url, data_root / f"{iso3}.zip", extract=True)

road_path = (
    data_root / f"data/{iso3}/openstreetmap/openstreetmap_roads-tertiary__{iso3}.gpkg"
)
flood_dir = data_root / f"data/{iso3}/jrc_floods"
flood_metadata_path = data_root / f"data/{iso3}/aqueduct_flood.csv"

flood_metadata = pd.read_csv(flood_metadata_path, keep_default_na=False)
required_columns = {"hazard", "epoch", "rcp", "gcm", "return_period", "path"}
missing_columns = required_columns - set(flood_metadata.columns)
if missing_columns:
    raise ValueError(f"Aqueduct metadata is missing columns: {sorted(missing_columns)}")

# Keep fluvial flooding to match the river-flood curves used in this example.
raster_manifest = flood_metadata.loc[flood_metadata["hazard"] == "fluvial"].copy()
if raster_manifest.empty:
    raise ValueError(f"No fluvial rasters found in {flood_metadata_path}")
raster_manifest["path"] = raster_manifest["path"].map(
    lambda path: data_root / f"data/{iso3}" / path
)
raster_manifest["epoch"] = pd.to_numeric(
    raster_manifest["epoch"], errors="raise"
).astype(int)
raster_manifest["return_period"] = pd.to_numeric(
    raster_manifest["return_period"], errors="raise"
).astype(int)
scenario_columns = ["rcp", "gcm", "epoch"]
raster_manifest["scenario"] = (
    raster_manifest[scenario_columns].astype(str).agg("|".join, axis=1)
)
raster_manifest = raster_manifest.sort_values(
    ["epoch", "rcp", "gcm", "return_period"]
).reset_index(drop=True)

if raster_manifest.duplicated(["scenario", "return_period"]).any():
    raise ValueError("Found duplicate rasters for a scenario/return-period pair")
flood_return_periods = tuple(sorted(raster_manifest["return_period"].unique()))
for scenario, records in raster_manifest.groupby("scenario", sort=False):
    if set(records["return_period"]) != set(flood_return_periods):
        raise ValueError(f"Incomplete return-period coverage for {scenario}")

scenario_metadata = (
    raster_manifest[["scenario", *scenario_columns]]
    .drop_duplicates()
    .sort_values(["epoch", "rcp", "gcm"])
    .reset_index(drop=True)
)
print(
    f"Found {len(raster_manifest)} fluvial rasters across "
    f"{len(scenario_metadata)} (RCP, GCM, epoch) scenarios."
)
scenario_metadata

In [ ]:
# Load every road feature, including roads with no modelled inundation.
roads = gpd.read_file(road_path, layer="lines")
if not roads["osm_id"].is_unique:
    raise ValueError("Expected a unique osm_id for every road asset")

# Recompute lengths on the shared split geometries in metres, then convert to km.
geod = Geod(ellps="WGS84")

# Calculate geodesic length in meters, then convert to km
roads["length_km"] = roads.geometry.apply(geod.geometry_length) / 1_000
road_length_km = roads.length_km.sum()
curve_metadata = {curve_id: get_metadata(curve_id) for curve_id in ("F7.2", "F7.3")}
curves = {curve_id: load_curve(curve_id) for curve_id in curve_metadata}

print(f"Road assets: {len(roads):,}; total length: {road_length_km:,.1f} km")
for curve_id, metadata in curve_metadata.items():
    print(
        f"{curve_id}: {metadata.hazard_name} hazard, "
        f"{metadata.exposed_element}, intensity unit {metadata.intensity_unit}"
    )
roads.head()

In [ ]:
scenario_frames = []
coverage_rows = []
intensity_columns = {f"flood_rp{rp}": f"flood_rp{rp}" for rp in flood_return_periods}

# Overlay each climate scenario separately because raster grids may differ.
for scenario, records in raster_manifest.groupby("scenario", sort=False):
    scenario_rasters = pd.DataFrame(
        {
            "path": records["path"].tolist(),
            "key": [f"flood_rp{rp}" for rp in records["return_period"]],
        }
    )
    scenario_splits = overlay_rasters(roads[["osm_id", "geometry"]], scenario_rasters)
    scenario_splits["length_km"] = (
        scenario_splits.geometry.apply(geod.geometry_length) / 1_000
    )
    scenario_splits["scenario"] = scenario
    assessed_length_km = scenario_splits["length_km"].sum()
    if scenario_splits["osm_id"].nunique() != len(roads):
        raise ValueError(f"Some road assets were lost during splitting for {scenario}")
    if not np.isclose(assessed_length_km, road_length_km, rtol=1e-5):
        raise ValueError(
            f"Road length was not conserved during splitting for {scenario}"
        )

    for record in records.itertuples():
        case = f"flood_rp{record.return_period}"
        with rasterio.open(record.path) as raster:
            nodata = raster.nodata
        depth = scenario_splits[case]
        missing = ~np.isfinite(depth)
        if nodata is not None:
            missing |= depth == nodata
        if ((depth < 0) & ~missing).any():
            raise ValueError(f"Unexpected negative flood depth in {record.path.name}")
        scenario_splits[case] = depth.where(~missing, 0.0)
        flooded = scenario_splits[case] > 0
        coverage_rows.append(
            {
                "scenario": scenario,
                "return_period": record.return_period,
                "n_assets": scenario_splits["osm_id"].nunique(),
                "assessed_length_km": assessed_length_km,
                "flooded_length_km": scenario_splits.loc[flooded, "length_km"].sum(),
            }
        )
    scenario_frames.append(
        scenario_splits[
            ["osm_id", "split", "scenario", "length_km", *intensity_columns]
        ].copy()
    )

damage_splits = pd.concat(scenario_frames, ignore_index=True)
coverage = pd.DataFrame(coverage_rows).merge(
    scenario_metadata, on="scenario", validate="many_to_one"
)
coverage_table = coverage.pivot(
    index=["rcp", "gcm", "epoch"],
    columns="return_period",
    values="flooded_length_km",
).round(2)
coverage_table

In [ ]:
result = calculate_direct_damages(
    damage_splits,
    asset_id="osm_id",
    scenario="scenario",
    quantity="length_km",
    intensity_columns=intensity_columns,
    return_periods={f"flood_rp{rp}": float(rp) for rp in flood_return_periods},
    curves=curves,
    curve_weights={"F7.2": 0.3, "F7.3": 0.7},
    cost_mean=300_000,
    cost_systemic_cv=0.25,
    hazard_cv=dict.fromkeys(intensity_columns, 0.1),
    eta_sd=0.1,
    n_samples=1_000,
    quantiles=None,
    ead_quantiles=(0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975),
    seed=42,
    aggregate_assets=True,
)

total_ead = (
    result.total_ead_quantiles.merge(
        scenario_metadata, on="scenario", validate="one_to_one"
    )
    .sort_values(["epoch", "rcp", "gcm"])
    .reset_index(drop=True)
)
if not (total_ead["n_assets"] == len(roads)).all():
    raise ValueError("The total EAD results do not include every road asset")
total_ead[
    [
        "scenario",
        "rcp",
        "gcm",
        "epoch",
        "n_assets",
        "n_splits",
        "q_02.5",
        "q_05",
        "q_25",
        "q_50",
        "q_75",
        "q_95",
        "q_97.5",
    ]
]

In [ ]:
historical = total_ead.loc[total_ead["rcp"] == "historical"]
future = total_ead.loc[total_ead["rcp"].isin(["4.5", "8.5"])]
if len(historical) != 1:
    raise ValueError("Expected one historical fluvial scenario for the reference line")

rcps = sorted(future["rcp"].unique(), key=float)
fig, axes = plt.subplots(1, len(rcps), figsize=(14, 5), sharey=True, squeeze=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
historical_result = historical.iloc[0]

for ax, rcp in zip(axes.flat, rcps):
    rcp_results = future.loc[future["rcp"] == rcp]
    for index, (gcm, gcm_results) in enumerate(rcp_results.groupby("gcm", sort=True)):
        gcm_results = gcm_results.sort_values("epoch")
        color = colors[index % len(colors)]
        epochs = gcm_results["epoch"].to_numpy()
        ax.fill_between(
            epochs,
            gcm_results["q_05"].to_numpy(),
            gcm_results["q_95"].to_numpy(),
            color=color,
            alpha=0.12,
        )
        ax.plot(
            epochs, gcm_results["q_50"].to_numpy(), marker="o", color=color, label=gcm
        )

    ax.axhspan(
        historical_result["q_05"],
        historical_result["q_95"],
        color="black",
        alpha=0.08,
    )
    ax.axhline(
        historical_result["q_50"],
        color="black",
        linestyle="--",
        label=f"Historical ({int(historical_result['epoch'])})",
    )
    epochs = sorted(rcp_results["epoch"].unique())
    ax.set_xticks(epochs, [str(epoch) for epoch in epochs])
    ax.set_title(f"RCP {rcp}")
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.25)
    ax.legend(title="GCM / reference", fontsize="small")

axes[0, 0].set_ylabel("Total expected annual damage (currency units/year)")
fig.suptitle(
    f"Road network EAD across Aqueduct flood scenarios ({len(roads):,} assets)"
)
fig.tight_layout()
fig